# Re-ingest manual curation and compare the rounds (Spyglass pipeline, step 4 of 4)

This notebook closes the loop. It:

1. **Re-ingests** the `curation_data.json` produced by the SpikeInterface GUI
   ([`Pipeline_Spyglass_ManualCuration.ipynb`](Pipeline_Spyglass_ManualCuration.ipynb)) back into
   Spyglass as a new curation (`curation_id = 2`).
2. **Compares** the three curation rounds — raw (`0`), automatic (`1`), and manual (`2`).
3. **Exposes** your chosen final curation to downstream pipelines via `SpikeSortingOutput`.

> **Environment:** run this notebook with the **`spyglass`** kernel. See [`README.md`](README.md)
> for the full chain.

## Connect to the database

In [ ]:
import json
from pathlib import Path
from pprint import pprint

import datajoint as dj
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import spikeinterface as si

# Load config for database connection info
dj_local_conf_path = "/Users/pauladkisson/Documents/CatalystNeuro/Spyglass/spyglass/dj_local_conf.json"
dj.config.load(dj_local_conf_path)

# Spyglass stores some parameters as native Python objects (dicts / lists) in the database;
# this flag lets DataJoint serialize and deserialize those blobs instead of rejecting them.
dj.config["enable_python_native_blobs"] = True

# General Spyglass imports (importing common connects to the database)
import spyglass.common as sgc
import spyglass.spikesorting.v1 as sgs
from spyglass.spikesorting.spikesorting_merge import SpikeSortingOutput
from spyglass.utils.nwb_helper_fn import get_nwb_copy_filename

## Parameters set manually

These must match the earlier notebooks.

In [ ]:
### Parameters set manually ###

# Session: the NWB copy that lives in the database (note the trailing underscore)
nwb_file_name = get_nwb_copy_filename("H3022-210806.nwb")  # -> "H3022-210806_.nwb"

# Which shank (sort group) and which epoch (interval) were sorted
sort_group_id = 0
interval_list_name = "01"  # first wake epoch for this session

# Pipeline parameters (must match the values used in Pipeline_Spyglass_SpikeSorting.ipynb)
preproc_param_name = "default"
sorter = "mountainsort5"
sorter_param_name = "default"

# Shared handoff folder for the manual-curation step (notebooks 2 -> 3 -> 4)
export_root = Path("manual_curation_export")

## Recover the sorting

In [ ]:
# Re-derive the sorting produced by Pipeline_Spyglass_SpikeSorting.ipynb from the parameters
# above. Every stage is keyed off the recording, so we first recover its recording_id, then the
# sorting_id, then point at the raw curation (curation_id = 0) that notebook registered.
recording_id = (
    sgs.SpikeSortingRecordingSelection
    & {
        "nwb_file_name": nwb_file_name,
        "sort_group_id": sort_group_id,
        "interval_list_name": interval_list_name,
        "preproc_param_name": preproc_param_name,
    }
).fetch1("recording_id")

sorting_id = (
    sgs.SpikeSortingSelection
    & {
        "recording_id": recording_id,
        "sorter": sorter,
        "sorter_param_name": sorter_param_name,
    }
).fetch1("sorting_id")

curation_key = {"sorting_id": sorting_id, "curation_id": 0}
assert sgs.CurationV1 & curation_key, (
    "No raw curation (curation_id = 0) found for this sorting. "
    "Run Pipeline_Spyglass_SpikeSorting.ipynb first."
)
sgs.CurationV1 & {"sorting_id": sorting_id}

## 1. Re-ingest the manual curation

The GUI writes curation in the SpikeInterface **v2 curation format**: `manual_labels` (per-unit
labels under a `"quality"` category of `good` / `noise` / `MUA`), `merges` (groups of unit ids),
and `removed` (units deleted as noise). Spyglass's `insert_curation` instead wants a
`labels = {unit_id: [label, ...]}` dict and `merge_groups = [[unit_id, ...], ...]`, where the valid
labels are `reject`, `noise`, `artifact`, `mua`, `accept`. The cell below translates between the
two and inserts the result as `curation_id = 2`, branching off the raw sort.

In [ ]:
export_dir = export_root / str(sorting_id)
curation_json_path = export_dir / "sorting_analyzer" / "spikeinterface_gui" / "curation_data.json"
assert curation_json_path.exists(), (
    f"No curation file at {curation_json_path}. "
    "Run Pipeline_Spyglass_ManualCuration.ipynb (step 3) and save in the GUI first."
)

with open(curation_json_path) as file:
    gui_curation = json.load(file)

# Map the GUI's quality labels onto Spyglass's valid labels.
LABEL_MAP = {"good": "accept", "noise": "noise", "MUA": "mua", "mua": "mua"}

labels = {}
for entry in gui_curation.get("manual_labels", []):
    unit_id = entry["unit_id"]
    flat = [
        LABEL_MAP.get(label, label)
        for label_list in entry["labels"].values()
        for label in label_list
    ]
    if flat:
        labels[unit_id] = flat

# Units removed in the GUI are treated as rejected.
for unit_id in gui_curation.get("removed", []):
    labels.setdefault(unit_id, []).append("reject")

merge_groups = [group["unit_ids"] for group in gui_curation.get("merges", [])]

print(f"manual labels: {labels}")
print(f"manual merge groups: {merge_groups}")

In [ ]:
sgs.CurationV1.insert_curation(
    sorting_id=sorting_id,
    parent_curation_id=0,
    labels=labels or None,
    merge_groups=merge_groups or None,
    description="after manual curation (SpikeInterface GUI)",
)
sgs.CurationV1 & {"sorting_id": sorting_id}

## 2. Compare the curation rounds

For each curation that exists, we summarize the unit count, how many units were labeled (and with
what), how many merge groups were defined, and the unit count after applying those merges.

In [ ]:
round_names = {0: "raw", 1: "automatic", 2: "manual"}
available_ids = sorted((sgs.CurationV1 & {"sorting_id": sorting_id}).fetch("curation_id"))

summary_rows = []
spike_times_by_round = {}
labels_by_round = {}
for curation_id in available_ids:
    key = {"sorting_id": sorting_id, "curation_id": curation_id}
    units = sgs.CurationV1.get_sorting(key, as_dataframe=True)

    # Per-unit labels (the raw sort has no curation_label column).
    if "curation_label" in units.columns:
        unit_labels = {idx: list(row) for idx, row in units["curation_label"].items()}
    else:
        unit_labels = {idx: [] for idx in units.index}
    labels_by_round[curation_id] = unit_labels

    label_tally = {}
    for unit_label_list in unit_labels.values():
        for label in unit_label_list:
            label_tally[label] = label_tally.get(label, 0) + 1

    merged_sorting = sgs.CurationV1.get_merged_sorting(key)
    n_merge_groups = len(gui_curation.get("merges", [])) if curation_id == 2 else None

    summary_rows.append(
        {
            "curation_id": curation_id,
            "round": round_names.get(curation_id, str(curation_id)),
            "n_units": len(units),
            "n_units_after_merge": len(merged_sorting.get_unit_ids()),
            "n_labeled": sum(1 for v in unit_labels.values() if v),
            "labels": label_tally,
        }
    )

    sorting_obj = sgs.CurationV1.get_sorting(key)
    spike_times_by_round[curation_id] = {
        unit_id: sorting_obj.get_unit_spike_train(unit_id, return_times=True)
        for unit_id in sorting_obj.get_unit_ids()
    }

pd.DataFrame(summary_rows).set_index("curation_id")

### Spike rasters by curation round

The same window for every round. Units that the round labeled `noise`/`reject` are drawn in grey;
all other units are colored. This makes the effect of each curation visible at a glance.

In [ ]:
raster_duration = 100.0  # seconds from the start of the sorted recording to display


def in_window(times, stop):
    times = np.asarray(times)
    return times[(times >= 0.0) & (times <= stop)]


fig, axes = plt.subplots(len(available_ids), 1, figsize=(15, 4 * len(available_ids)), sharex=True)
axes = np.atleast_1d(axes)
for ax, curation_id in zip(axes, available_ids):
    spike_times = spike_times_by_round[curation_id]
    unit_labels = labels_by_round[curation_id]
    unit_ids = list(spike_times.keys())
    windowed = [in_window(spike_times[unit_id], raster_duration) for unit_id in unit_ids]
    colors = [
        "0.7"
        if any(label in ("noise", "reject") for label in unit_labels.get(unit_id, []))
        else "tab:blue"
        for unit_id in unit_ids
    ]
    ax.eventplot(windowed, linelengths=0.8, linewidths=0.8, colors=colors)
    ax.set_yticks(range(len(unit_ids)))
    ax.set_yticklabels(unit_ids)
    ax.set_ylabel("Unit")
    ax.set_title(f"curation_id {curation_id} ({round_names.get(curation_id, curation_id)})")
axes[-1].set_xlabel("Seconds since start of sorted recording")
plt.tight_layout()

## 3. Expose the chosen curation downstream

Pick which curation should be the one downstream pipelines use, and insert it into the
`SpikeSortingOutput` merge table. Each `(sorting_id, curation_id)` becomes its own `merge_id`, so
this does not disturb the raw `curation_id = 0` entry that step 1 already inserted — multiple
curations of the same sorting can coexist there.

In [ ]:
# Choose the final curation to expose: 0 (raw), 1 (automatic), or 2 (manual).
final_curation_id = 2

final_curation_key = {"sorting_id": sorting_id, "curation_id": final_curation_id}
SpikeSortingOutput.insert(
    (sgs.CurationV1 & final_curation_key).fetch("KEY", as_dict=True),
    part_name="CurationV1",
)
SpikeSortingOutput.merge_view()

### Removing the raw entry (optional)

If you would rather *replace* the raw `curation_id = 0` entry in `SpikeSortingOutput` than keep it
alongside the curated one, delete it by cascading from `CurationV1` upstream and then sweeping the
orphaned merge-table master — do **not** call `.delete()` on the merge master directly (it raises a
`force_parts` error). The reset cell in `Pipeline_Spyglass_SpikeSorting.ipynb` is the template:

```python
raw_merge_id = (
    SpikeSortingOutput.CurationV1 & {"sorting_id": sorting_id, "curation_id": 0}
).fetch1("merge_id")
(sgs.CurationV1 & {"sorting_id": sorting_id, "curation_id": 0}).delete()  # cascades to the part row
(SpikeSortingOutput & {"merge_id": raw_merge_id}).super_delete(warn=False, safemode=False)
```